In [1]:
import os

In [2]:
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net\\notebooks'

In [3]:
os.chdir('../')
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net'

In [7]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir :Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [8]:
%pwd

'c:\\Users\\yonas\\Desktop\\Mlops_Data_Science\\wine-quality-elastic-net'

In [16]:
! pip install  joblib

In [17]:
! pip install ensure

In [18]:
! pip install python-box

In [36]:
from src.constants.path import PARAMS_FILE_PATH,SCHEMA_FILE_PATH,CONFIG_FILE_PATH
from src.utils.common import read_yaml , create_directories

In [41]:
class ConfigurationManager:
    def __init__(self , config_filepath =CONFIG_FILE_PATH, 
                 params_filepath=PARAMS_FILE_PATH , schema_filepath =SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_roots])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config= DataIngestionConfig(
             root_dir = config.root_dir, 
             source_URL= config.source_URL,
             local_data_file = config.local_data_file,
             unzip_dir = config.unzip_dir
        )
        return data_ingestion_config
   

In [38]:
import os
import urllib.request as request
from src.logger.logger import  logger
import zipfile

In [39]:
## component-Data Ingestion

class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config=config
    
    # Downloading the zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)


In [42]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-19 17:41:08,697] [INFO] [src.logger.logger] : yaml file: config\config.yaml loaded successfully
[2026-03-19 17:41:08,698] [INFO] [src.logger.logger] : yaml file: params.yaml loaded successfully
[2026-03-19 17:41:08,707] [INFO] [src.logger.logger] : yaml file: schema.yaml loaded successfully
[2026-03-19 17:41:08,710] [INFO] [src.logger.logger] : created directory at: artifacts
[2026-03-19 17:41:08,712] [INFO] [src.logger.logger] : created directory at: artifacts/data_ingestion
[2026-03-19 17:41:09,598] [INFO] [src.logger.logger] : artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-